In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q20/annotated/checkpoints/post_cell_4.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

date1 = pd.Timestamp("1996-01-01")
date2 = pd.Timestamp("1997-01-01")
psel = part.P_NAME.str.startswith("azure")
nsel = nation.N_NAME == "JORDAN"
lsel = (lineitem.L_SHIPDATE >= date1) & (lineitem.L_SHIPDATE < date2)
fpart = part[psel]
fnation = nation[nsel]
flineitem = lineitem[lsel]
jn1 = fpart.merge(partsupp, left_on="P_PARTKEY", right_on="PS_PARTKEY")
jn2 = jn1.merge(
    flineitem,
    left_on=["PS_PARTKEY", "PS_SUPPKEY"],
    right_on=["L_PARTKEY", "L_SUPPKEY"],
)
gb = jn2.groupby(
    ["PS_PARTKEY", "PS_SUPPKEY", "PS_AVAILQTY"], as_index=False, sort=False
)["L_QUANTITY"].sum()
gbsel = gb.PS_AVAILQTY > (0.5 * gb.L_QUANTITY)
fgb = gb[gbsel]
jn3 = fgb.merge(supplier, left_on="PS_SUPPKEY", right_on="S_SUPPKEY")
jn4 = fnation.merge(jn3, left_on="N_NATIONKEY", right_on="S_NATIONKEY")
jn4 = jn4.loc[:, ["S_NAME", "S_ADDRESS"]]
total = jn4.sort_values("S_NAME").drop_duplicates()


In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q20/annotated/checkpoints/post_cell_5.pickle